# MAD1 - Unidad 4: ANOVA y Diseños Experimentales
## Práctica en Python

**Métodos de Análisis de Datos 1 - UNS**  
Primer Semestre 2026

## Librerías necesarias

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg  # pip install pingouin

# Configuración de gráficos
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

---
# 1. ANOVA de Un Factor

## Datos de ejemplo: Dietas de pollos

In [ ]:
# Peso ganado (gramos) con 4 dietas
data_dieta = {
    'Peso': [179,160,136,227,217,168,108,124,  # Dieta 1
             309,229,181,141,260,203,          # Dieta 2
             141,148,169,213,257,244,271,      # Dieta 3
             309,229,181,141,260,203,148],     # Dieta 4
    'Dieta': ['D1']*8 + ['D2']*6 + ['D3']*7 + ['D4']*7
}

df_dieta = pd.DataFrame(data_dieta)
print(df_dieta.groupby('Dieta')['Peso'].describe())

## ANOVA

In [ ]:
# Método 1: scipy (más simple)
grupos = [df_dieta[df_dieta['Dieta']==d]['Peso'].values for d in ['D1','D2','D3','D4']]
f_stat, p_valor = stats.f_oneway(*grupos)
print(f"F = {f_stat:.3f}, p-valor = {p_valor:.6f}")

# Método 2: statsmodels (tabla ANOVA completa)
modelo1 = ols('Peso ~ C(Dieta)', data=df_dieta).fit()
tabla_anova1 = sm.stats.anova_lm(modelo1, typ=2)
print("\nTabla ANOVA:")
print(tabla_anova1)

## Supuestos

In [ ]:
# Residuos
residuos = modelo1.resid

# 1. NORMALIDAD
stat_sw, p_sw = stats.shapiro(residuos)
print(f"Shapiro-Wilk: W = {stat_sw:.4f}, p-valor = {p_sw:.4f}")

# Gráfico Q-Q
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
stats.probplot(residuos, dist="norm", plot=ax1)
ax1.set_title("Gráfico Q-Q")

# Histograma
ax2.hist(residuos, bins=10, edgecolor='black')
ax2.set_xlabel("Residuos")
ax2.set_ylabel("Frecuencia")
ax2.set_title("Histograma de Residuos")
plt.tight_layout()
plt.show()

In [ ]:
# 2. HOMOCEDASTICIDAD
stat_lev, p_lev = stats.levene(*grupos)
print(f"Levene: F = {stat_lev:.4f}, p-valor = {p_lev:.4f}")

# Gráfico residuos vs ajustados
plt.figure(figsize=(8, 5))
plt.scatter(modelo1.fittedvalues, residuos, alpha=0.6)
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
plt.xlabel("Valores Ajustados")
plt.ylabel("Residuos")
plt.title("Residuos vs Valores Ajustados")
plt.grid(True, alpha=0.3)
plt.show()

## Comparaciones múltiples

In [ ]:
# Tukey HSD con pingouin
posthoc1 = pg.pairwise_tukey(data=df_dieta, dv='Peso', between='Dieta')
print(posthoc1[['A', 'B', 'mean(A)', 'mean(B)', 'diff', 'p-tukey']])

---
# 2. ANOVA de Dos Factores

## Datos: Rendimiento de cultivos

In [ ]:
# Rendimiento (kg/ha) según Fertilizante y Riego
data_cultivo = {
    'Fertilizante': ['F1']*8 + ['F2']*8 + ['F3']*8,
    'Riego': ['Bajo']*4 + ['Alto']*4 + ['Bajo']*4 + ['Alto']*4 + ['Bajo']*4 + ['Alto']*4,
    'Rendimiento': [45,47,46,48, 55,57,56,58,
                    52,54,53,55, 58,60,59,61,
                    48,50,49,51, 62,64,63,65]
}

df_cultivo = pd.DataFrame(data_cultivo)
print(df_cultivo.groupby(['Fertilizante', 'Riego'])['Rendimiento'].mean())

## ANOVA con interacción

In [ ]:
# Modelo con interacción
modelo2 = ols('Rendimiento ~ C(Fertilizante) + C(Riego) + C(Fertilizante):C(Riego)', 
              data=df_cultivo).fit()

tabla_anova2 = sm.stats.anova_lm(modelo2, typ=2)
print(tabla_anova2)

**Interpretación:** Las tres pruebas son significativas. Como hay interacción significativa (p < 0.001), el efecto de un factor depende del nivel del otro.

## Gráfico de interacción

In [ ]:
# Calcular medias
medias_cultivo = df_cultivo.groupby(['Fertilizante', 'Riego'])['Rendimiento'].mean().reset_index()

# Gráfico
plt.figure(figsize=(10, 6))
for fert in ['F1', 'F2', 'F3']:
    datos_fert = medias_cultivo[medias_cultivo['Fertilizante'] == fert]
    plt.plot(datos_fert['Riego'], datos_fert['Rendimiento'], 
             marker='o', label=fert, linewidth=2, markersize=10)

plt.xlabel('Nivel de Riego', fontsize=12)
plt.ylabel('Rendimiento Medio (kg/ha)', fontsize=12)
plt.title('Gráfico de Interacción', fontsize=14)
plt.legend(title='Fertilizante', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Observación:** Las líneas no son paralelas → confirma interacción. F3 es mejor con riego alto, F2 es más estable.

## Comparaciones post-hoc (efectos simples)

In [ ]:
# Crear variable de combinación
df_cultivo['Combinacion'] = df_cultivo['Fertilizante'] + '_' + df_cultivo['Riego']

# Comparar todas las combinaciones
posthoc2 = pg.pairwise_tukey(data=df_cultivo, dv='Rendimiento', between='Combinacion')
print(posthoc2[['A', 'B', 'mean(A)', 'mean(B)', 'diff', 'p-tukey']].head(10))

---
# 3. Diseño en Bloques (DBCA)

## Datos: Variedades de trigo

In [ ]:
# Rendimiento (ton/ha) de 4 variedades en 5 lotes
data_trigo = {
    'Variedad': ['V1']*5 + ['V2']*5 + ['V3']*5 + ['V4']*5,
    'Lote': list(range(1,6)) * 4,
    'Rendimiento': [5.2,4.8,5.5,4.9,5.1,
                    6.1,5.7,6.4,5.8,6.0,
                    5.8,5.4,6.1,5.5,5.7,
                    5.5,5.1,5.8,5.2,5.4]
}

df_trigo = pd.DataFrame(data_trigo)
df_trigo['Lote'] = df_trigo['Lote'].astype('category')

## ANOVA en bloques

In [ ]:
# Modelo: tratamiento + bloque (sin interacción)
modelo_dbca = ols('Rendimiento ~ C(Variedad) + C(Lote)', data=df_trigo).fit()
tabla_dbca = sm.stats.anova_lm(modelo_dbca, typ=2)
print(tabla_dbca)

**Interpretación:** Tanto Variedad como Lote son significativos. El bloqueo fue efectivo.

## Comparaciones múltiples (solo variedades)

In [ ]:
posthoc_dbca = pg.pairwise_tukey(data=df_trigo, dv='Rendimiento', between='Variedad')
print(posthoc_dbca[['A', 'B', 'mean(A)', 'mean(B)', 'diff', 'p-tukey']])

---
# 4. Diseño en Cuadrado Latino (DCL)

## Datos: Tiempo de secado de pintura

In [ ]:
# 4 formulaciones, 4 operadores, 4 días
data_dcl = {
    'Tiempo': [73,74,71,75, 73,75,72,74, 75,73,76,74, 72,75,73,71],
    'Formulacion': list('ABCDBCDACDABDABC'),
    'Operador': [1,1,1,1, 2,2,2,2, 3,3,3,3, 4,4,4,4],
    'Dia': [1,2,3,4]*4
}

df_dcl = pd.DataFrame(data_dcl)
df_dcl['Operador'] = df_dcl['Operador'].astype('category')
df_dcl['Dia'] = df_dcl['Dia'].astype('category')

## ANOVA en Cuadrado Latino

In [ ]:
# Modelo: tratamiento + fila + columna (sin interacciones)
modelo_dcl = ols('Tiempo ~ C(Formulacion) + C(Operador) + C(Dia)', 
                 data=df_dcl).fit()
tabla_dcl = sm.stats.anova_lm(modelo_dcl, typ=2)
print(tabla_dcl)

**Interpretación:** Probar si Formulación es significativa controlando por Operador y Día.

---
# 5. Verificación de Supuestos (Ejemplo completo)

In [ ]:
# Usar modelo de dos factores como ejemplo
modelo = modelo2
residuos = modelo.resid

# 1. NORMALIDAD
stat_sw, p_sw = stats.shapiro(residuos)
print(f"Shapiro-Wilk: W = {stat_sw:.4f}, p-valor = {p_sw:.4f}")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Q-Q plot
stats.probplot(residuos, dist="norm", plot=axes[0,0])
axes[0,0].set_title("Gráfico Q-Q de Normalidad")

# Histograma
axes[0,1].hist(residuos, bins=10, edgecolor='black')
axes[0,1].set_xlabel("Residuos")
axes[0,1].set_ylabel("Frecuencia")
axes[0,1].set_title("Histograma de Residuos")

# 2. HOMOCEDASTICIDAD - Residuos vs ajustados
axes[1,0].scatter(modelo.fittedvalues, residuos, alpha=0.6)
axes[1,0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1,0].set_xlabel("Valores Ajustados")
axes[1,0].set_ylabel("Residuos")
axes[1,0].set_title("Residuos vs Valores Ajustados")
axes[1,0].grid(True, alpha=0.3)

# 3. INDEPENDENCIA - Residuos vs orden
axes[1,1].plot(range(len(residuos)), residuos, 'o-', alpha=0.6)
axes[1,1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1,1].set_xlabel("Orden de Observación")
axes[1,1].set_ylabel("Residuos")
axes[1,1].set_title("Residuos vs Orden")
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Prueba de Levene
grupos_cultivo = [df_cultivo[df_cultivo['Fertilizante']==f]['Rendimiento'] 
                  for f in ['F1','F2','F3']]
stat_lev, p_lev = stats.levene(*grupos_cultivo)
print(f"Levene: F = {stat_lev:.4f}, p-valor = {p_lev:.4f}")

# Durbin-Watson (requiere statsmodels)
from statsmodels.stats.stattools import durbin_watson
dw_stat = durbin_watson(residuos)
print(f"Durbin-Watson: {dw_stat:.4f} (cercano a 2 = no autocorrelación)")

---
# 6. Transformaciones

## Box-Cox

In [ ]:
from scipy.stats import boxcox

# Transformación Box-Cox (requiere datos positivos)
y_trans, lambda_opt = boxcox(df_dieta['Peso'])
print(f"Lambda óptimo: {lambda_opt:.4f}")

# Agregar columna transformada
df_dieta['Peso_BC'] = y_trans

# Re-analizar con datos transformados
modelo_bc = ols('Peso_BC ~ C(Dieta)', data=df_dieta).fit()
tabla_bc = sm.stats.anova_lm(modelo_bc, typ=2)
print("\nANOVA con datos transformados:")
print(tabla_bc)

# Verificar normalidad nuevamente
residuos_bc = modelo_bc.resid
stat_bc, p_bc = stats.shapiro(residuos_bc)
print(f"\nShapiro-Wilk (datos transformados): W = {stat_bc:.4f}, p = {p_bc:.4f}")

## Otras transformaciones comunes

In [ ]:
# Raíz cuadrada (conteos, varianza proporcional a media)
# np.sqrt(y)

# Logaritmo (varianza proporcional a media^2)
# np.log(y)  o  np.log10(y)

# Arcoseno (proporciones/porcentajes)
# np.arcsin(np.sqrt(y))

---
# 7. Pruebas No Paramétricas

## Kruskal-Wallis (alternativa a ANOVA un factor)

In [ ]:
# Cuando no se cumplen supuestos de normalidad
grupos_kw = [df_dieta[df_dieta['Dieta']==d]['Peso'].values 
             for d in ['D1','D2','D3','D4']]

h_stat, p_kw = stats.kruskal(*grupos_kw)
print(f"Kruskal-Wallis: H = {h_stat:.3f}, p-valor = {p_kw:.6f}")

# Comparaciones post-hoc con pingouin
posthoc_kw = pg.pairwise_tests(data=df_dieta, dv='Peso', between='Dieta',
                                parametric=False, padjust='bonf')
print("\nComparaciones post-hoc (Bonferroni):")
print(posthoc_kw[['A', 'B', 'p-unc', 'p-corr']])

## Friedman (alternativa a DBCA)

In [ ]:
# Organizar datos en matriz: bloques (filas) x tratamientos (columnas)
# Ejemplo: 5 bloques x 4 variedades
matriz_trigo = df_trigo.pivot(index='Lote', columns='Variedad', values='Rendimiento').values

stat_friedman, p_friedman = stats.friedmanchisquare(*matriz_trigo.T)
print(f"Friedman: χ² = {stat_friedman:.3f}, p-valor = {p_friedman:.6f}")

---
# 8. Ejercicios Propuestos

1. **DCA:** Comparar 5 métodos de enseñanza. Simular datos y realizar ANOVA completo.

2. **DBCA:** Experimento agrícola con 3 fertilizantes en 6 parcelas. Verificar si el bloqueo fue necesario.

3. **DCL:** Diseñar un cuadrado latino 5×5 para estudiar 5 tratamientos controlando dos fuentes de variación.

4. **Interacciones:** Crear un dataset donde haya interacción significativa. Graficar e interpretar.

5. **Transformaciones:** Generar datos con varianza no constante. Aplicar Box-Cox y comparar resultados.

---

## Notas Finales

- **Siempre verificar supuestos** antes de interpretar resultados
- **Si hay interacción significativa**, no interpretar efectos principales por separado
- **Ajustar por comparaciones múltiples** (Tukey, Bonferroni, etc.)
- **Reportar tamaños de efecto** además de p-valores (eta², Cohen's d)
- **Documentar el proceso** de aleatorización y diseño

---

**Referencias:**

- Montgomery, D.C. (2017). *Design and Analysis of Experiments*.
- Statsmodels documentation: https://www.statsmodels.org/
- Pingouin documentation: https://pingouin-stats.org/